# CloudSeekers — ETL + Forecast + Risco de OLA (v4)

MVP preliminar da Sprint 3 com pipeline reproduzível:

**Bronze → Silver → Forecast D+1/D+7 → Risco de OLA → Gold → Azure SQL**

### Objetivos desta versão
- padronizar a estrutura de pastas do repositório;
- exportar as duas tabelas Gold exatamente no formato esperado pelo Azure SQL;
- separar métricas e figuras usadas como evidência no PPT;
- manter o notebook responsável por Data Science/ML e deixar a carga no Azure para um script `.py` independente.


## 0. Configuração do projeto


In [ ]:
from pathlib import Path
import shutil
import warnings
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.rcParams['figure.dpi'] = 110

# ------------------------------------------------------------------
# Estrutura de diretórios do repositório
# ------------------------------------------------------------------
# Funciona se o notebook for executado:
# 1) a partir da raiz do repositório; ou
# 2) dentro da pasta notebooks/.
cwd = Path.cwd().resolve()

if cwd.name.lower() == 'notebooks':
    PROJECT_ROOT = cwd.parent
elif (cwd / 'notebooks').exists() or (cwd / '.git').exists():
    PROJECT_ROOT = cwd
else:
    # Fallback para execução isolada/Colab/ambiente de avaliação.
    PROJECT_ROOT = cwd / 'cloudseekers-aiops'

DATA_DIR = PROJECT_ROOT / 'data'
BRONZE_DIR = DATA_DIR / 'bronze'
SILVER_DIR = DATA_DIR / 'silver'
GOLD_DIR = DATA_DIR / 'gold'

GOLD_FORECAST_DIR = GOLD_DIR / 'forecast'
GOLD_RISK_DIR = GOLD_DIR / 'risco_ola'

OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
METRICS_DIR = OUTPUTS_DIR / 'metrics'
DOCS_DIR = OUTPUTS_DIR / 'docs'

for d in [
    BRONZE_DIR,
    SILVER_DIR,
    GOLD_FORECAST_DIR,
    GOLD_RISK_DIR,
    FIGURES_DIR,
    METRICS_DIR,
    DOCS_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Dataset original
# ------------------------------------------------------------------
# Procura o dataset em locais comuns.
candidates = [
    BRONZE_DIR / 'LW-DATASET.xlsx',
    PROJECT_ROOT / 'LW-DATASET.xlsx',
    cwd / 'LW-DATASET.xlsx',
    Path('/content/LW-DATASET.xlsx'),
    Path('/mnt/data/LW-DATASET.xlsx'),
]

SOURCE_XLSX = next((p for p in candidates if p.exists()), candidates[0])
BRONZE_FILE = BRONZE_DIR / 'LW-DATASET.xlsx'

if not SOURCE_XLSX.exists():
    raise FileNotFoundError(
        'LW-DATASET.xlsx não encontrado. Coloque o arquivo na raiz do projeto '
        'ou em data/bronze/ e execute novamente.'
    )

# Bronze preserva o arquivo original sem transformação.
if SOURCE_XLSX.resolve() != BRONZE_FILE.resolve() and not BRONZE_FILE.exists():
    shutil.copy2(SOURCE_XLSX, BRONZE_FILE)

print('CloudSeekers — configuração carregada')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Bronze:       {BRONZE_FILE}')
print(f'Silver:       {SILVER_DIR}')
print(f'Gold Forecast:{GOLD_FORECAST_DIR}')
print(f'Gold Risco:   {GOLD_RISK_DIR}')
print(f'Figuras:      {FIGURES_DIR}')
print(f'Métricas:     {METRICS_DIR}')


## 1. Ingestão — Camada Bronze

O arquivo original fornecido pela Locaweb é preservado **sem alteração** na camada Bronze. Essa decisão garante rastreabilidade e permite reproduzir todo o tratamento posteriormente.


In [ ]:
df_bronze = pd.read_excel(BRONZE_FILE, sheet_name='Dataset Geral')
n_bronze = len(df_bronze)

print(f'Volume de registros (Bronze): {n_bronze:,}')
print(f'Quantidade de colunas: {df_bronze.shape[1]}')
print(f'Período bruto: {pd.to_datetime(df_bronze["Aberto"]).min()} -> {pd.to_datetime(df_bronze["Aberto"]).max()}')
df_bronze.head(3)

## 2. Qualidade de Dados — Antes da Limpeza


In [ ]:
valid_priorities = {'1 - Crítica', '2 - Alta', '3 - Média', '4 - Baixa', '5 - Muito Baixa'}

qa_before = {
    'volume_bronze': n_bronze,
    'duplicados_numero': int(df_bronze['Número'].duplicated().sum()),
    'nulos_por_coluna': df_bronze.isna().sum().to_dict(),
    'prioridades_fora_dominio': int((~df_bronze['Prioridade'].isin(valid_priorities)).sum()),
}

aberto_tmp = pd.to_datetime(df_bronze['Aberto'], errors='coerce')
resolvido_tmp = pd.to_datetime(df_bronze['Resolvido'], errors='coerce')
encerrado_tmp = pd.to_datetime(df_bronze['Encerrado'], errors='coerce')
qa_before['datas_inconsistentes'] = int(((resolvido_tmp < aberto_tmp) | (encerrado_tmp < aberto_tmp)).sum())

print(f"Volume Bronze:             {qa_before['volume_bronze']:,}")
print(f"Duplicados por Número:     {qa_before['duplicados_numero']}")
print(f"Datas inconsistentes:      {qa_before['datas_inconsistentes']}")
print(f"Prioridades fora domínio:  {qa_before['prioridades_fora_dominio']}")
print('\nNulos relevantes:')
for col, n in qa_before['nulos_por_coluna'].items():
    if n > 0:
        print(f'  {col}: {n:,} ({n/n_bronze:.1%})')

## 3. ETL — Bronze → Silver

O tratamento mantém as regras do dicionário de dados e evita alterar semanticamente o dataset. Campos condicionais permanecem nulos quando o próprio processo de negócio prevê ausência de valor.


In [ ]:
df = df_bronze.copy()

# 1) Padronização de strings
for c in df.select_dtypes(include=['object', 'string']).columns:
    df[c] = df[c].astype('string').str.strip()

# 2) Prioridade — código e rótulo derivados
prioridade_cod = pd.to_numeric(df['Prioridade'].str.extract(r'^(\d)')[0], errors='coerce')
df['Prioridade_Cod'] = prioridade_cod.astype('Int64')
df['Prioridade_Label'] = df['Prioridade'].str.extract(r'-\s*(.*)$')[0]

# 3) Datas
for c in ['Aberto', 'Resolvido', 'Encerrado']:
    df[c] = pd.to_datetime(df[c], errors='coerce')

# 4) Categorias opcionais: torna a ausência explícita
for c in ['Produto', 'Categoria', 'Subcategoria', 'Item de configuração']:
    df[c] = df[c].fillna('Não informado')

# 5) Flags auxiliares
# Importante: são calculadas após o strip, mantendo valores ausentes do negócio.
df['Tem_Incidente_Pai'] = df['Incidente Pai'].notna()
df['Possui_Solucao_Registrada'] = df['Solução'].notna()
df['Possui_Codigo_Fechamento'] = df['Código de fechamento'].notna()

# 6) KPI
for c in ['Entrou para KPI?', 'KPI Violado?']:
    df[c] = df[c].str.upper().str.strip()

df['KPI_Violado_Bool'] = df['KPI Violado?'].map({'SIM': True, 'NAO': False}).astype('boolean')

# Regra observada no dataset: quando Entrou para KPI = NAO, KPI Violado? deve permanecer nulo.
regra_kpi_quebrada = int(((df['Entrou para KPI?'] == 'NAO') & df['KPI Violado?'].notna()).sum())
print(f"Violações da regra KPI: {regra_kpi_quebrada}")
assert regra_kpi_quebrada == 0, 'Regra de negócio de KPI violada — revisar antes de publicar a Silver.'

# 7) Duplicados pela chave de negócio
n_dup = int(df.duplicated(subset=['Número']).sum())
df = df.drop_duplicates(subset=['Número']).reset_index(drop=True)
print(f'Duplicados removidos: {n_dup}')

## 4. Validação da Duração


In [ ]:
# O dicionário define Duração como o tempo entre abertura e resolução (ou encerramento).
# Validamos a consistência sem sobrescrever o campo oficial fornecido pela Locaweb.
duracao_oficial = pd.to_numeric(df['Duração'], errors='coerce')
duracao_calculada = np.where(
    df['Resolvido'].notna(),
    (df['Resolvido'] - df['Aberto']).dt.total_seconds(),
    (df['Encerrado'] - df['Aberto']).dt.total_seconds(),
)

duracao_calculada = pd.Series(duracao_calculada, index=df.index, dtype='float64')
mask_dur = duracao_oficial.notna() & duracao_calculada.notna()
dif_dur = (duracao_oficial[mask_dur] - duracao_calculada[mask_dur]).abs()

pct_exato = float((dif_dur < 1).mean())
pct_60s = float((dif_dur <= 60).mean())
corr_dur = float(duracao_oficial[mask_dur].corr(duracao_calculada[mask_dur]))

print(f'Linhas validadas: {mask_dur.sum():,}')
print(f'Correspondência exata (<1s): {pct_exato:.2%}')
print(f'Diferença de até 60s:        {pct_60s:.2%}')
print(f'Correlação:                   {corr_dur:.6f}')

# Flag de auditoria; o campo oficial é preservado.
df['Duracao_Divergente_60s'] = False
df.loc[mask_dur, 'Duracao_Divergente_60s'] = dif_dur > 60
print(f'Divergências > 60s: {int(df["Duracao_Divergente_60s"].sum()):,}')

## 5. Qualidade de Dados — Depois da Limpeza


In [ ]:
qa_after = {
    'volume_bronze': n_bronze,
    'volume_silver': len(df),
    'registros_removidos': n_bronze - len(df),
    'duplicados_numero': int(df['Número'].duplicated().sum()),
    'regra_kpi_violada': regra_kpi_quebrada,
    'prioridades_fora_dominio': int((~df['Prioridade'].isin(valid_priorities)).sum()),
    'duracao_correlacao': corr_dur,
    'duracao_exata_pct': pct_exato,
    'duracao_ate_60s_pct': pct_60s,
}

print(pd.Series(qa_after))
print('\nNulos remanescentes — preservados quando condicionais ao negócio:')
for c, n in df.isna().sum().items():
    if n > 0:
        print(f'  {c}: {n:,} ({n/len(df):.1%})')

## 6. Publicação da Silver


In [ ]:
silver_csv = SILVER_DIR / 'incidents_silver.csv'
df.to_csv(silver_csv, index=False)
print(f'Silver CSV: {silver_csv}')

# Parquet é preferível para uma arquitetura de Data Lake, mas depende de pyarrow/fastparquet.
silver_parquet = SILVER_DIR / 'incidents_silver.parquet'
try:
    df.to_parquet(silver_parquet, index=False)
    print(f'Silver Parquet: {silver_parquet}')
except Exception as e:
    print('Parquet não gerado neste ambiente. CSV continua disponível.')
    print('Opcional: instale pyarrow com `pip install pyarrow`.')

print(f'Linhas: {len(df):,} | Colunas: {df.shape[1]}')

## 7. Dicionário de Dados da Silver


In [ ]:
lines = [
    '# Dicionário de Dados — Camada Silver',
    '',
    'Base tratada usada no pipeline de forecast P2/P3.',
    '',
    '| Campo | Tipo | Descrição / regra |',
    '|---|---|---|',
    '| Número | string | Chave de negócio; duplicados removidos |',
    '| Prioridade | string | Prioridade original padronizada |',
    '| Prioridade_Cod | int | Código numérico derivado da prioridade |',
    '| Prioridade_Label | string | Rótulo derivado da prioridade |',
    '| Produto / Categoria / Subcategoria / Item de configuração | string | Nulos convertidos em Não informado |',
    '| Aberto / Resolvido / Encerrado | datetime | Tipagem de marcos temporais |',
    '| Duração | numérico | Campo oficial preservado; validado contra cálculo temporal |',
    '| Duracao_Divergente_60s | bool | Flag de auditoria para diferença superior a 60s |',
    '| Entrou para KPI? | SIM/NAO | Indicador oficial padronizado |',
    '| KPI Violado? | SIM/NAO/null | Indicador oficial; nulo quando não entra no KPI |',
    '| KPI_Violado_Bool | boolean | Versão booleana derivada |',
    '',
    '## Evidências de qualidade',
    f'- Volume Bronze: {n_bronze:,}',
    f'- Volume Silver: {len(df):,}',
    f'- Registros removidos: {n_bronze-len(df)}',
    f'- Violações da regra KPI: {regra_kpi_quebrada}',
    f'- Duração com correspondência exata: {pct_exato:.2%}',
    f'- Duração com diferença de até 60s: {pct_60s:.2%}',
    f'- Correlação da Duração com cálculo temporal: {corr_dur:.6f}',
]

path_dict = DOCS_DIR / 'data_dictionary.md'
path_dict.write_text('\n'.join(lines), encoding='utf-8')
print(f'Dicionário salvo em: {path_dict}')

## 8. Diagnóstico da Série Temporal e Mudança de Regime

A base apresenta uma forte mudança de patamar durante 2025. Em vez de chamar todo o ano de “regime estável”, o notebook agora **mede e documenta** essa mudança.

O forecast continua usando 2025 para preservar histórico suficiente, mas adiciona variáveis de regime ao modelo. Essa decisão evita descartar meses úteis e, ao mesmo tempo, deixa explícito que setembro/2025 representa uma alteração operacional importante.


In [ ]:
df['Data'] = df['Aberto'].dt.floor('D')
alvo_prioridades = {'2 - Alta': 'P2', '3 - Média': 'P3'}
sub = df[df['Prioridade'].isin(alvo_prioridades)].copy()
sub['P'] = sub['Prioridade'].map(alvo_prioridades)

daily = sub.groupby(['Data', 'P']).size().unstack(fill_value=0)
full_idx = pd.date_range(daily.index.min(), daily.index.max(), freq='D')
daily = daily.reindex(full_idx, fill_value=0)
daily.index.name = 'Data'

TRAINING_START = pd.Timestamp('2025-01-01')
REGIME_DATE = pd.Timestamp('2025-09-01')
daily_model = daily.loc[TRAINING_START:].copy()

monthly = daily_model.resample('MS').agg(['sum', 'mean'])
print('Volume mensal de P2/P3 em 2025:')
display(monthly.round(1))

pre = daily_model.loc[:REGIME_DATE - pd.Timedelta(days=1)].mean()
post = daily_model.loc[REGIME_DATE:].mean()
regime_summary = pd.DataFrame({
    'Media_Diaria_Pre_Set': pre,
    'Media_Diaria_Pos_Set': post,
    'Razao_Pos_Pre': post / pre,
}).round(2)
print('\nComparação pré/pós setembro/2025:')
display(regime_summary)

monthly_totals = daily_model.resample('MS').sum()
ax = monthly_totals.plot(figsize=(11, 4), marker='o', title='Volume mensal P2/P3 — diagnóstico de mudança de regime')
ax.axvline(REGIME_DATE, linestyle='--', linewidth=1, label='Mudança de regime — set/2025')
ax.set_xlabel('Mês')
ax.set_ylabel('Incidentes')
ax.legend()
plt.tight_layout()
plt.savefig(GOLD_DIR / 'diagnostico_regime_mensal.png', dpi=150, bbox_inches='tight')
plt.show()

daily_model.to_csv(SILVER_DIR / 'daily_volume_by_priority.csv')

## 9. Feature Engineering Temporal

Para cada prioridade, o forecast usa apenas informações disponíveis até a data de referência.

Features principais:

- volume do dia de referência;
- lags de 1 a 7, 14, 21 e 28 dias;
- médias, medianas e desvios móveis de 7, 14 e 28 dias;
- calendário da data de referência;
- calendário da data prevista;
- flag e distância temporal da mudança de regime.

São criados alvos diretos para **D+1, D+2, ..., D+7**. Isso permite representar a próxima semana inteira, em vez de tratar D+7 apenas como um único dia isolado.


In [ ]:
HORIZONS = list(range(1, 8))
LAGS = [1, 2, 3, 4, 5, 6, 7, 14, 21, 28]
ROLL_WINDOWS = [7, 14, 28]


def build_master_features(series: pd.Series) -> pd.DataFrame:
    t = pd.DataFrame(index=series.index)
    t['current'] = series
    t['ref_dow'] = t.index.dayofweek
    t['ref_month'] = t.index.month
    t['ref_weekend'] = (t['ref_dow'] >= 5).astype(int)

    for lag in LAGS:
        t[f'lag_{lag}'] = series.shift(lag)

    for w in ROLL_WINDOWS:
        # Janela incluindo a data de referência, pois a previsão é gerada após a consolidação diária.
        t[f'roll_mean_{w}'] = series.rolling(w).mean()
        t[f'roll_median_{w}'] = series.rolling(w).median()
        t[f'roll_std_{w}'] = series.rolling(w).std()

    t['regime_flag'] = (t.index >= REGIME_DATE).astype(int)
    t['days_since_regime'] = np.maximum((t.index - REGIME_DATE).days, 0)

    for h in HORIZONS:
        target_date = t.index + pd.to_timedelta(h, unit='D')
        t[f'target_dow_h{h}'] = target_date.dayofweek
        t[f'target_month_h{h}'] = target_date.month
        t[f'target_weekend_h{h}'] = (target_date.dayofweek >= 5).astype(int)
        t[f'target_h{h}'] = series.shift(-h)

    return t

BASE_FEATURES = [
    'current', 'ref_dow', 'ref_month', 'ref_weekend',
    *[f'lag_{x}' for x in LAGS],
    *[f'roll_mean_{x}' for x in ROLL_WINDOWS],
    *[f'roll_median_{x}' for x in ROLL_WINDOWS],
    *[f'roll_std_{x}' for x in ROLL_WINDOWS],
    'regime_flag', 'days_since_regime'
]

feature_tables = {p: build_master_features(daily_model[p]) for p in ['P2', 'P3']}

for p, table in feature_tables.items():
    table.to_csv(SILVER_DIR / f'features_{p}.csv')
    print(p, table.shape)

## 10. Baselines e Modelos Candidatos

O notebook não assume que XGBoost será automaticamente superior. Ele compara **Persistência, Sazonal 7 dias, Média móvel 7 dias, Random Forest e XGBoost**.

Para reduzir complexidade e manter o MVP eficiente, os horizontes D+1...D+7 são organizados em formato longo e os modelos de ML aprendem o horizonte como uma feature. Assim, cada prioridade utiliza um modelo por família em cada fold, em vez de sete modelos independentes.


In [ ]:
def make_xgb():
    return XGBRegressor(
        n_estimators=220,
        max_depth=2,
        learning_rate=0.04,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=5,
        objective='reg:absoluteerror',
        random_state=42,
        n_jobs=2,
    )


def make_rf():
    return RandomForestRegressor(
        n_estimators=150,
        max_depth=7,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=2,
    )


def baseline_prediction(X, model_name):
    h = X['horizon'].astype(int)
    if model_name == 'Persistence':
        return X['current'].to_numpy()
    if model_name == 'RollingMean7':
        return X['roll_mean_7'].to_numpy()
    if model_name == 'Seasonal7':
        # y(t+h-7): para h=7 usa o volume atual; para h<7 usa lag_(7-h).
        out = []
        for _, row in X.iterrows():
            hh = int(row['horizon'])
            out.append(row['current'] if hh == 7 else row[f'lag_{7-hh}'])
        return np.asarray(out, dtype=float)
    raise ValueError(model_name)


def calc_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    denom = np.abs(y_true).sum()
    wape = (np.abs(y_true - y_pred).sum() / denom * 100) if denom > 0 else np.nan
    return mae, rmse, wape


def to_long_table(master: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for h in HORIZONS:
        cols = BASE_FEATURES + [f'target_dow_h{h}', f'target_month_h{h}', f'target_weekend_h{h}', f'target_h{h}']
        tmp = master[cols].copy()
        tmp = tmp.rename(columns={
            f'target_dow_h{h}': 'target_dow',
            f'target_month_h{h}': 'target_month',
            f'target_weekend_h{h}': 'target_weekend',
            f'target_h{h}': 'target',
        })
        tmp['horizon'] = h
        tmp['Data_Referencia'] = tmp.index
        tmp['Data_Previsao'] = tmp.index + pd.to_timedelta(h, unit='D')
        parts.append(tmp)
    return pd.concat(parts, axis=0, ignore_index=True).dropna().sort_values(['Data_Referencia', 'horizon'])

MODEL_FEATURES = BASE_FEATURES + ['target_dow', 'target_month', 'target_weekend', 'horizon']
CANDIDATES = ['Persistence', 'Seasonal7', 'RollingMean7', 'RandomForest', 'XGBoost']
long_tables = {p: to_long_table(feature_tables[p]) for p in ['P2', 'P3']}
for p, lt in long_tables.items():
    print(p, lt.shape, lt['Data_Referencia'].min().date(), '->', lt['Data_Referencia'].max().date())

## 11. Validação Temporal — Rolling-Origin Backtest

A validação usa quatro janelas temporais sucessivas de 30 dias. Em cada fold, o treinamento utiliza apenas alvos cuja **Data_Previsao ocorre antes da janela validada**, evitando vazamento temporal.

O desempenho é medido separadamente para D+1...D+7 e o campeão de cada horizonte é escolhido pelo menor MAE médio entre os folds.


In [ ]:
VALIDATION_WINDOW = 30
N_FOLDS = 4
MIN_TRAIN_DAYS = 90

metric_records = []
pred_records = []

for p in ['P2', 'P3']:
    data_long = long_tables[p].copy()
    ref_dates = pd.Index(sorted(data_long['Data_Referencia'].unique()))

    # Quatro janelas de 30 dias no período mais recente.
    fold_starts = []
    for k in range(N_FOLDS, 0, -1):
        pos = len(ref_dates) - k * VALIDATION_WINDOW
        if pos >= MIN_TRAIN_DAYS:
            fold_starts.append(pos)

    for fold_id, start_pos in enumerate(fold_starts, start=1):
        val_dates = ref_dates[start_pos:start_pos + VALIDATION_WINDOW]
        if len(val_dates) == 0:
            continue
        val_start = pd.Timestamp(val_dates.min())
        val_end = pd.Timestamp(val_dates.max())

        # Evita leakage: o alvo do treino precisa ocorrer antes da primeira referência validada.
        train = data_long[data_long['Data_Previsao'] < val_start].copy()
        valid = data_long[data_long['Data_Referencia'].isin(val_dates)].copy()

        X_train, y_train = train[MODEL_FEATURES], train['target']
        X_valid, y_valid = valid[MODEL_FEATURES], valid['target']

        candidate_predictions = {
            'Persistence': baseline_prediction(X_valid, 'Persistence'),
            'Seasonal7': baseline_prediction(X_valid, 'Seasonal7'),
            'RollingMean7': baseline_prediction(X_valid, 'RollingMean7'),
        }

        rf = make_rf()
        rf.fit(X_train, y_train)
        candidate_predictions['RandomForest'] = np.clip(rf.predict(X_valid), 0, None)

        xgb = make_xgb()
        xgb.fit(X_train, y_train)
        candidate_predictions['XGBoost'] = np.clip(xgb.predict(X_valid), 0, None)

        for model_name, pred in candidate_predictions.items():
            valid_out = valid[['Data_Referencia', 'Data_Previsao', 'horizon', 'target']].copy()
            valid_out['pred'] = pred

            for h in HORIZONS:
                hmask = valid_out['horizon'] == h
                if not hmask.any():
                    continue
                yh = valid_out.loc[hmask, 'target']
                ph = valid_out.loc[hmask, 'pred']
                mae, rmse, wape = calc_metrics(yh, ph)
                metric_records.append({
                    'Prioridade': p,
                    'Horizonte_Dias': h,
                    'Horizonte': f'D+{h}',
                    'Fold': fold_id,
                    'Modelo': model_name,
                    'MAE': mae,
                    'RMSE': rmse,
                    'WAPE_pct': wape,
                    'Inicio_Validacao': val_start,
                    'Fim_Validacao': val_end,
                    'N_Validacao': int(hmask.sum()),
                })

            for row, pr in zip(valid.itertuples(index=False), pred):
                pred_records.append({
                    'Data_Referencia': row.Data_Referencia,
                    'Data_Previsao': row.Data_Previsao,
                    'Prioridade': p,
                    'Horizonte_Dias': int(row.horizon),
                    'Horizonte': f'D+{int(row.horizon)}',
                    'Fold': fold_id,
                    'Modelo': model_name,
                    'Volume_Real': float(row.target),
                    'Volume_Previsto': float(max(pr, 0)),
                })

metrics_cv = pd.DataFrame(metric_records)
preds_cv = pd.DataFrame(pred_records)

summary_cv = (
    metrics_cv.groupby(['Prioridade', 'Horizonte_Dias', 'Horizonte', 'Modelo'], as_index=False)
    .agg(MAE=('MAE', 'mean'), RMSE=('RMSE', 'mean'), WAPE_pct=('WAPE_pct', 'mean'), Folds=('Fold', 'nunique'))
)

champions = (
    summary_cv.sort_values(['Prioridade', 'Horizonte_Dias', 'MAE'])
    .groupby(['Prioridade', 'Horizonte_Dias'], as_index=False)
    .first()
    .rename(columns={'Modelo': 'Modelo_Selecionado', 'MAE': 'MAE_CV', 'RMSE': 'RMSE_CV', 'WAPE_pct': 'WAPE_CV_pct'})
)

display(champions[['Prioridade', 'Horizonte', 'Modelo_Selecionado', 'MAE_CV', 'RMSE_CV', 'WAPE_CV_pct']].round(2))

## 12. Comparação dos Modelos


In [ ]:
comparison = summary_cv.sort_values(['Prioridade', 'Horizonte_Dias', 'MAE']).copy()
display(comparison.round(2))

# Arquivos de avaliação ficam em outputs/metrics.
comparison.to_csv(METRICS_DIR / 'metricas_forecast_todos_modelos.csv', index=False)
champions.to_csv(METRICS_DIR / 'modelos_campeoes_forecast.csv', index=False)

# Evidência compacta para PPT: D+1 e D+7.
evidence = comparison[comparison['Horizonte_Dias'].isin([1, 7])].copy()
evidence.to_csv(METRICS_DIR / 'metricas_forecast_d1_d7.csv', index=False)
display(evidence.round(2))


## 13. Backtest do Modelo Selecionado

As previsões abaixo são **out-of-sample** dentro das janelas do rolling-origin backtest. Isso significa que cada previsão foi gerada sem utilizar o valor futuro correspondente durante o treinamento daquele fold.


In [ ]:
champ_lookup = champions.set_index(
    ['Prioridade', 'Horizonte_Dias']
)['Modelo_Selecionado'].to_dict()

preds_champion = preds_cv[
    preds_cv.apply(
        lambda r: r['Modelo'] == champ_lookup[(r['Prioridade'], r['Horizonte_Dias'])],
        axis=1
    )
].copy()

preds_champion['Volume_Real'] = preds_champion['Volume_Real'].round().astype(int)
preds_champion['Volume_Previsto'] = preds_champion['Volume_Previsto'].round(1)

preds_champion.to_csv(
    GOLD_FORECAST_DIR / 'fct_previsao_backtest.csv',
    index=False
)

# Figuras individuais P2/P3.
for p in ['P2', 'P3']:
    d = preds_champion[
        (preds_champion['Prioridade'] == p)
        & (preds_champion['Horizonte_Dias'] == 1)
    ].sort_values('Data_Previsao')

    plt.figure(figsize=(11, 4))
    plt.plot(d['Data_Previsao'], d['Volume_Real'], label='Real')
    plt.plot(d['Data_Previsao'], d['Volume_Previsto'], label='Previsto')
    plt.title(f'{p} — Rolling backtest D+1 | Campeão: {champ_lookup[(p, 1)]}')
    plt.xlabel('Data prevista')
    plt.ylabel('Incidentes')
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR / f'forecast_real_vs_previsto_{p}.png',
        dpi=160,
        bbox_inches='tight'
    )
    plt.show()

# Evidência única para o PPT: P2 + P3 no D+1.
d_plot = preds_champion[preds_champion['Horizonte_Dias'] == 1].copy()

real_daily = (
    d_plot.groupby('Data_Previsao', as_index=False)['Volume_Real']
    .sum()
    .sort_values('Data_Previsao')
)

pred_daily = (
    d_plot.groupby('Data_Previsao', as_index=False)['Volume_Previsto']
    .sum()
    .sort_values('Data_Previsao')
)

plt.figure(figsize=(12, 5))
plt.plot(real_daily['Data_Previsao'], real_daily['Volume_Real'], label='Real')
plt.plot(pred_daily['Data_Previsao'], pred_daily['Volume_Previsto'], label='Previsto')
plt.title('CloudSeekers — Volume P2 + P3 | Real x Previsto (D+1)')
plt.xlabel('Data prevista')
plt.ylabel('Incidentes')
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / 'forecast_real_vs_previsto.png',
    dpi=180,
    bbox_inches='tight'
)
plt.show()


## 14. D+7 como Próxima Semana — Avaliação Agregada

Além das sete previsões diárias, o MVP calcula o **volume total esperado nos próximos 7 dias**. A soma é feita por data de referência e prioridade.


In [ ]:
weekly_backtest = (
    preds_champion.groupby(['Data_Referencia', 'Prioridade'], as_index=False)
    .agg(
        Volume_Real_7d=('Volume_Real', 'sum'),
        Volume_Previsto_7d=('Volume_Previsto', 'sum'),
    )
)

weekly_metrics = []
for p in ['P2', 'P3']:
    d = weekly_backtest[weekly_backtest['Prioridade'] == p]
    mae, rmse, wape = calc_metrics(d['Volume_Real_7d'], d['Volume_Previsto_7d'])
    weekly_metrics.append({
        'Prioridade': p,
        'Horizonte': 'Próximos 7 dias',
        'MAE_7d': mae,
        'RMSE_7d': rmse,
        'WAPE_7d_pct': wape,
        'N_Referencias': len(d),
    })

weekly_metrics = pd.DataFrame(weekly_metrics)
display(weekly_metrics.round(2))

weekly_metrics.to_csv(
    METRICS_DIR / 'metricas_forecast_semana.csv',
    index=False
)

weekly_backtest.to_csv(
    GOLD_FORECAST_DIR / 'fct_backtest_proxima_semana.csv',
    index=False
)


## 15. Treinamento Final e Inferência Futura

Depois de validar e selecionar o melhor método por prioridade e horizonte, o notebook volta a utilizar todo o histórico disponível com alvo conhecido e gera previsões **após a última data observada**.

Isso resolve uma limitação do notebook original, que realizava backtest, mas não produzia uma previsão futura efetiva para o MVP.


In [ ]:
def fit_final_ml(model_name, X, y):
    if model_name == 'XGBoost':
        model = make_xgb()
    elif model_name == 'RandomForest':
        model = make_rf()
    else:
        return None
    model.fit(X, y)
    return model

future_rows = []
final_models = {}

for p in ['P2', 'P3']:
    master = feature_tables[p]
    latest_ref = master.index.max()
    latest = master.loc[latest_ref]
    hist_long = long_tables[p]

    # Treina, no máximo, um modelo final de cada família por prioridade.
    needed_ml = {champ_lookup[(p, h)] for h in HORIZONS if champ_lookup[(p, h)] in ['XGBoost', 'RandomForest']}
    fitted = {}
    for model_name in needed_ml:
        model = fit_final_ml(model_name, hist_long[MODEL_FEATURES], hist_long['target'])
        fitted[model_name] = model
        final_models[(p, model_name)] = model

    for h in HORIZONS:
        model_name = champ_lookup[(p, h)]
        target_date = latest_ref + pd.Timedelta(days=h)

        row = {f: latest[f] for f in BASE_FEATURES}
        row['target_dow'] = target_date.dayofweek
        row['target_month'] = target_date.month
        row['target_weekend'] = int(target_date.dayofweek >= 5)
        row['horizon'] = h
        X_future = pd.DataFrame([row], columns=MODEL_FEATURES)

        if model_name in fitted:
            pred = float(np.clip(fitted[model_name].predict(X_future)[0], 0, None))
        else:
            pred = float(baseline_prediction(X_future, model_name)[0])

        future_rows.append({
            'Data_Referencia': latest_ref,
            'Data_Previsao': target_date,
            'Prioridade': p,
            'Horizonte_Dias': h,
            'Horizonte': f'D+{h}',
            'Modelo_Selecionado': model_name,
            'Volume_Previsto': round(pred, 1),
        })

future_forecast = pd.DataFrame(future_rows).sort_values(['Prioridade', 'Data_Previsao'])
display(future_forecast)

future_week = (
    future_forecast.groupby(['Data_Referencia', 'Prioridade'], as_index=False)
    .agg(
        Inicio_Semana=('Data_Previsao', 'min'),
        Fim_Semana=('Data_Previsao', 'max'),
        Volume_Previsto_7d=('Volume_Previsto', 'sum'),
    )
)
future_week['Volume_Previsto_7d'] = future_week['Volume_Previsto_7d'].round(1)
print('\nResumo — próxima semana:')
display(future_week)

## 16. Visualização da Previsão Futura


In [ ]:
# Figuras por prioridade.
for p in ['P2', 'P3']:
    d = future_forecast[future_forecast['Prioridade'] == p]
    plt.figure(figsize=(9, 4))
    plt.plot(d['Data_Previsao'], d['Volume_Previsto'], marker='o')
    plt.title(f'{p} — Previsão diária para os próximos 7 dias')
    plt.xlabel('Data prevista')
    plt.ylabel('Volume previsto')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR / f'forecast_d1_d7_{p}.png',
        dpi=160,
        bbox_inches='tight'
    )
    plt.show()

# Figura única para o PPT.
pivot_future = (
    future_forecast
    .pivot_table(
        index='Data_Previsao',
        columns='Prioridade',
        values='Volume_Previsto',
        aggfunc='sum'
    )
    .sort_index()
)

plt.figure(figsize=(10, 5))
for p in ['P2', 'P3']:
    if p in pivot_future.columns:
        plt.plot(
            pivot_future.index,
            pivot_future[p],
            marker='o',
            label=p
        )

plt.title('CloudSeekers — Forecast P2/P3 | D+1 a D+7')
plt.xlabel('Data prevista')
plt.ylabel('Volume previsto')
plt.xticks(rotation=30)
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / 'forecast_d1_d7.png',
    dpi=180,
    bbox_inches='tight'
)
plt.show()


## 17. Camada Gold — Forecast

A saída principal desta etapa é preparada **exatamente no formato esperado pela tabela `gold.fct_previsao_futura` do Azure SQL**.

Campos publicados:

- `data_referencia`
- `data_previsao`
- `prioridade`
- `horizonte_dias`
- `volume_previsto`
- `modelo`

Arquivos auxiliares de backtest permanecem na Gold para rastreabilidade, enquanto métricas e gráficos são gravados em `outputs/`.


In [ ]:
# ------------------------------------------------------------------
# GOLD PRINCIPAL — formato exato para Azure SQL
# ------------------------------------------------------------------
forecast_sql = future_forecast.rename(columns={
    'Data_Referencia': 'data_referencia',
    'Data_Previsao': 'data_previsao',
    'Prioridade': 'prioridade',
    'Horizonte_Dias': 'horizonte_dias',
    'Volume_Previsto': 'volume_previsto',
    'Modelo_Selecionado': 'modelo',
})[
    [
        'data_referencia',
        'data_previsao',
        'prioridade',
        'horizonte_dias',
        'volume_previsto',
        'modelo',
    ]
].copy()

forecast_sql['data_referencia'] = pd.to_datetime(
    forecast_sql['data_referencia']
).dt.date

forecast_sql['data_previsao'] = pd.to_datetime(
    forecast_sql['data_previsao']
).dt.date

forecast_sql['horizonte_dias'] = forecast_sql['horizonte_dias'].astype(int)
forecast_sql['volume_previsto'] = forecast_sql['volume_previsto'].astype(float)

forecast_sql.to_csv(
    GOLD_FORECAST_DIR / 'fct_previsao_futura.csv',
    index=False
)

# Saídas analíticas complementares.
future_week.to_csv(
    GOLD_FORECAST_DIR / 'fct_previsao_proxima_semana.csv',
    index=False
)

summary_cv.to_csv(
    METRICS_DIR / 'metricas_forecast.csv',
    index=False
)

print('Gold Forecast pronta para Azure SQL:')
display(forecast_sql)

print('\nArquivo principal:')
print(GOLD_FORECAST_DIR / 'fct_previsao_futura.csv')


## 18. Modelo de Risco de OLA — Definição do Problema

O segundo motor analítico do CloudSeekers estima o **risco de violação de OLA** para incidentes P2 e P3.

### Regra de modelagem

- O modelo é treinado somente com incidentes que **Entraram para KPI = SIM**.
- O alvo oficial é o campo **KPI Violado?** fornecido pela Locaweb.
- P2 e P3 são priorizados por serem obrigatórios no desafio.
- Variáveis conhecidas apenas após o encerramento do incidente **não entram como features**, evitando leakage.

### Campos deliberadamente excluídos do modelo

`Duração`, `Resolvido`, `Encerrado`, `Código de fechamento`, `Solução`, `Status` e o próprio `KPI Violado?`.

Esses campos carregam informação posterior ao momento em que o alerta deveria ser gerado.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score, recall_score, f1_score, fbeta_score,
    average_precision_score, roc_auc_score, confusion_matrix
)
from xgboost import XGBClassifier

# Base operacional de P2/P3 para gerar contexto histórico de volume.
risk_priorities = ['2 - Alta', '3 - Média']

ops = df[df['Prioridade'].isin(risk_priorities)].copy()
ops['Data'] = ops['Aberto'].dt.floor('D')

daily_ops = (
    ops.groupby(['Data', 'Prioridade'])
    .size()
    .rename('Volume_Dia')
    .reset_index()
)

all_dates = pd.date_range(daily_ops['Data'].min(), daily_ops['Data'].max(), freq='D')
grid = pd.MultiIndex.from_product(
    [all_dates, risk_priorities],
    names=['Data', 'Prioridade']
).to_frame(index=False)

daily_ops = (
    grid.merge(daily_ops, on=['Data', 'Prioridade'], how='left')
    .fillna({'Volume_Dia': 0})
    .sort_values(['Prioridade', 'Data'])
)

# Somente informação anterior à data do incidente.
daily_ops['Volume_D1'] = daily_ops.groupby('Prioridade')['Volume_Dia'].shift(1)
daily_ops['Media_7d'] = daily_ops.groupby('Prioridade')['Volume_Dia'].transform(
    lambda s: s.shift(1).rolling(7, min_periods=1).mean()
)
daily_ops['Media_14d'] = daily_ops.groupby('Prioridade')['Volume_Dia'].transform(
    lambda s: s.shift(1).rolling(14, min_periods=1).mean()
)

risk = df[
    df['Prioridade'].isin(risk_priorities)
    & (df['Entrou para KPI?'] == 'SIM')
].copy()

risk['Target_OLA'] = (risk['KPI Violado?'] == 'SIM').astype(int)
risk['Data'] = risk['Aberto'].dt.floor('D')

risk = risk.merge(
    daily_ops[['Data', 'Prioridade', 'Volume_D1', 'Media_7d', 'Media_14d']],
    on=['Data', 'Prioridade'],
    how='left'
)

# Features temporais disponíveis no momento da abertura.
risk['Mes'] = risk['Aberto'].dt.month
risk['DiaSemana'] = risk['Aberto'].dt.dayofweek
risk['Hora'] = risk['Aberto'].dt.hour
risk['FimSemana'] = (risk['DiaSemana'] >= 5).astype(int)
risk['PeriodoDia'] = pd.cut(
    risk['Hora'],
    bins=[-1, 5, 11, 17, 23],
    labels=['Madrugada', 'Manhã', 'Tarde', 'Noite']
).astype('string')

print(f'Incidentes elegíveis P2/P3: {len(risk):,}')
print(f'Violações de OLA: {risk["Target_OLA"].sum():,}')
print(f'Taxa de violação: {risk["Target_OLA"].mean():.2%}')


## 19. Desbalanceamento da Classe

A violação de OLA é um evento raro. Por isso, **acurácia não é utilizada como métrica principal**.

O modelo será avaliado principalmente por:

- **PR-AUC (Average Precision)** — adequada para classes raras;
- **Recall** — capacidade de encontrar violações reais;
- **Precision** — proporção de alertas que realmente violaram;
- **F1 / F2** — equilíbrio entre Precision e Recall, com F2 dando maior peso ao Recall;
- **Lift no Top 5%** — concentração de violações entre os incidentes com maior score de risco.


In [ ]:
risk_summary = (
    risk.groupby('Prioridade')['Target_OLA']
    .agg(Incidentes='count', Violacoes='sum', Taxa='mean')
    .reset_index()
)
risk_summary['Taxa'] = (risk_summary['Taxa'] * 100).round(2)
display(risk_summary)

plt.figure(figsize=(7, 4))
counts = risk['Target_OLA'].value_counts().sort_index()
plt.bar(['Não violou', 'Violou'], counts.values)
plt.title('Distribuição do alvo — KPI Violado?')
plt.ylabel('Quantidade de incidentes')
plt.tight_layout()
plt.savefig(GOLD_DIR / 'ola_distribuicao_target.png', dpi=150, bbox_inches='tight')
plt.show()


## 20. Feature Engineering do Risco de OLA

As features foram limitadas a informações conhecidas na abertura do incidente ou provenientes do histórico anterior.

### Features categóricas
- Prioridade
- Produto
- Categoria
- Subcategoria
- Grupo designado
- Item de configuração
- Aberto por
- Período do dia

### Features numéricas / temporais
- Mês
- Dia da semana
- Hora
- Fim de semana
- Volume do dia anterior
- Média de volume dos últimos 7 dias
- Média de volume dos últimos 14 dias

Categorias muito raras são agrupadas para reduzir overfitting e dimensionalidade.


In [ ]:
risk_cat_features = [
    'Prioridade', 'Produto', 'Categoria', 'Subcategoria',
    'Grupo designado', 'Item de configuração', 'Aberto por', 'PeriodoDia'
]
risk_num_features = [
    'Mes', 'DiaSemana', 'Hora', 'FimSemana',
    'Volume_D1', 'Media_7d', 'Media_14d'
]
risk_features = risk_cat_features + risk_num_features

# Split temporal: treino → validação → teste fora do tempo.
VAL_START = pd.Timestamp('2025-09-01')
TEST_START = pd.Timestamp('2025-11-01')

risk_train = risk[risk['Aberto'] < VAL_START].copy()
risk_val = risk[(risk['Aberto'] >= VAL_START) & (risk['Aberto'] < TEST_START)].copy()
risk_test = risk[risk['Aberto'] >= TEST_START].copy()

print('Treino:', len(risk_train), '| Violações:', int(risk_train['Target_OLA'].sum()))
print('Validação:', len(risk_val), '| Violações:', int(risk_val['Target_OLA'].sum()))
print('Teste OOT:', len(risk_test), '| Violações:', int(risk_test['Target_OLA'].sum()))

# Agrupa categorias raras usando SOMENTE o conjunto de treino.
MIN_CATEGORY_COUNT = 20
rare_maps = {}

for col in risk_cat_features:
    s = risk_train[col].astype('string').fillna('Não informado')
    counts = s.value_counts()
    kept = set(counts[counts >= MIN_CATEGORY_COUNT].index.astype(str))
    rare_maps[col] = kept

def apply_rare_mapping(frame):
    out = frame.copy()
    for col, kept in rare_maps.items():
        s = out[col].astype('string').fillna('Não informado')
        out[col] = s.where(s.isin(kept), '__RARE__')
    return out

risk_train_m = apply_rare_mapping(risk_train)
risk_val_m = apply_rare_mapping(risk_val)
risk_test_m = apply_rare_mapping(risk_test)


## 21. Modelos Candidatos para Risco de OLA

São comparados dois modelos:

1. **Regressão Logística balanceada** — baseline interpretável;
2. **XGBoost Classifier** — modelo não linear capaz de capturar interações entre variáveis.

A escolha do campeão é feita pelo **PR-AUC na validação temporal**.


In [ ]:
risk_cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=10))
])

risk_num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

risk_preprocess = ColumnTransformer([
    ('cat', risk_cat_pipe, risk_cat_features),
    ('num', risk_num_pipe, risk_num_features),
])

risk_logit = Pipeline([
    ('prep', risk_preprocess),
    ('clf', LogisticRegression(
        max_iter=1200,
        class_weight='balanced',
        solver='liblinear',
        random_state=42
    ))
])

n_pos = int(risk_train_m['Target_OLA'].sum())
n_neg = int(len(risk_train_m) - n_pos)
scale_pos_weight = n_neg / max(n_pos, 1)

risk_xgb = Pipeline([
    ('prep', risk_preprocess),
    ('clf', XGBClassifier(
        n_estimators=350,
        max_depth=4,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.80,
        min_child_weight=3,
        reg_lambda=2.0,
        objective='binary:logistic',
        eval_metric='aucpr',
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=4
    ))
])

risk_candidates = {
    'LogisticRegression_Balanced': risk_logit,
    'XGBoost_Classifier': risk_xgb,
}

risk_val_results = []
risk_val_probs = {}

Xtr = risk_train_m[risk_features]
ytr = risk_train_m['Target_OLA']
Xv = risk_val_m[risk_features]
yv = risk_val_m['Target_OLA']

for model_name, model in risk_candidates.items():
    model.fit(Xtr, ytr)
    prob = model.predict_proba(Xv)[:, 1]
    risk_val_probs[model_name] = prob

    risk_val_results.append({
        'Modelo': model_name,
        'PR_AUC_Validacao': average_precision_score(yv, prob),
        'ROC_AUC_Validacao': roc_auc_score(yv, prob),
    })

risk_model_comparison = pd.DataFrame(risk_val_results).sort_values(
    'PR_AUC_Validacao', ascending=False
)

display(risk_model_comparison.round(4))

risk_champion_name = risk_model_comparison.iloc[0]['Modelo']
risk_champion = risk_candidates[risk_champion_name]
print('Modelo campeão:', risk_champion_name)


## 22. Definição do Threshold Operacional

Como o evento é raro, usar automaticamente `0,50` pode não representar a melhor regra de alerta.

O threshold é selecionado **somente na validação**, maximizando o **F2 Score**, que dá mais peso ao Recall.

O conjunto de teste permanece isolado até a avaliação final.


In [ ]:
val_prob = risk_val_probs[risk_champion_name]

threshold_rows = []
for threshold in np.linspace(0.05, 0.95, 91):
    pred = (val_prob >= threshold).astype(int)
    threshold_rows.append({
        'Threshold': threshold,
        'Precision': precision_score(yv, pred, zero_division=0),
        'Recall': recall_score(yv, pred, zero_division=0),
        'F1': f1_score(yv, pred, zero_division=0),
        'F2': fbeta_score(yv, pred, beta=2, zero_division=0),
        'Alertas': int(pred.sum()),
    })

threshold_table = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_table.sort_values(
    ['F2', 'Recall', 'Precision'],
    ascending=False
).iloc[0]

RISK_THRESHOLD = float(best_threshold_row['Threshold'])

print(f'Threshold selecionado: {RISK_THRESHOLD:.2f}')
display(best_threshold_row.to_frame('Resultado').T.round(4))

threshold_table.to_csv(GOLD_DIR / 'ola_threshold_validacao.csv', index=False)


## 23. Avaliação Final — Teste Fora do Tempo

O teste utiliza os meses mais recentes e não participa da escolha do modelo nem do threshold.

Além das métricas tradicionais, calculamos o **Lift no Top 5% de risco**. Esse indicador responde:

> “Se a operação analisar primeiro os 5% de incidentes com maior score, quantas vezes a concentração de violações é maior que a média da base?”


In [ ]:
Xt = risk_test_m[risk_features]
yt = risk_test_m['Target_OLA']

test_prob = risk_champion.predict_proba(Xt)[:, 1]
test_pred = (test_prob >= RISK_THRESHOLD).astype(int)

pr_auc_test = average_precision_score(yt, test_prob)
roc_auc_test = roc_auc_score(yt, test_prob)
precision_test = precision_score(yt, test_pred, zero_division=0)
recall_test = recall_score(yt, test_pred, zero_division=0)
f1_test = f1_score(yt, test_pred, zero_division=0)
f2_test = fbeta_score(yt, test_pred, beta=2, zero_division=0)
cm = confusion_matrix(yt, test_pred)

# Top 5% de maior risco.
rank_df = pd.DataFrame({
    'Target': yt.to_numpy(),
    'Score': test_prob
}).sort_values('Score', ascending=False)

top_n = max(1, int(np.ceil(len(rank_df) * 0.05)))
top5 = rank_df.head(top_n)

base_rate = rank_df['Target'].mean()
top5_rate = top5['Target'].mean()
lift_top5 = top5_rate / base_rate if base_rate > 0 else np.nan
recall_top5 = top5['Target'].sum() / max(rank_df['Target'].sum(), 1)

risk_metrics_test = pd.DataFrame([{
    'Modelo': risk_champion_name,
    'Threshold': RISK_THRESHOLD,
    'PR_AUC': pr_auc_test,
    'ROC_AUC': roc_auc_test,
    'Precision': precision_test,
    'Recall': recall_test,
    'F1': f1_test,
    'F2': f2_test,
    'Lift_Top5pct': lift_top5,
    'Recall_Top5pct': recall_top5,
    'Taxa_Base': base_rate,
    'N_Teste': len(risk_test),
    'Violacoes_Teste': int(yt.sum()),
}])

display(risk_metrics_test.round(4))

cm_df = pd.DataFrame(
    cm,
    index=['Real_NAO', 'Real_SIM'],
    columns=['Prev_NAO', 'Prev_SIM']
)

print('Matriz de confusão:')
display(cm_df)

# Métricas para documentação/PPT.
risk_metrics_test.to_csv(
    METRICS_DIR / 'metricas_risco_ola.csv',
    index=False
)

cm_df.to_csv(
    METRICS_DIR / 'matriz_confusao_ola.csv'
)

# Figura da matriz de confusão.
plt.figure(figsize=(5.5, 4.5))
plt.imshow(cm)
plt.title('CloudSeekers — Matriz de Confusão | Risco de OLA')
plt.xticks([0, 1], ['Prev. NÃO', 'Prev. SIM'])
plt.yticks([0, 1], ['Real NÃO', 'Real SIM'])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, int(cm[i, j]), ha='center', va='center')

plt.xlabel('Predição')
plt.ylabel('Real')
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / 'matriz_confusao_ola.png',
    dpi=180,
    bbox_inches='tight'
)
plt.show()


## 24. Explicabilidade — Importância das Features

Para o XGBoost, extraímos as variáveis com maior importância no modelo.

> A importância indica associação preditiva dentro do modelo; não deve ser interpretada automaticamente como causalidade.


In [ ]:
feature_importance = pd.DataFrame()

if risk_champion_name == 'XGBoost_Classifier':
    prep_fitted = risk_champion.named_steps['prep']
    clf_fitted = risk_champion.named_steps['clf']

    names = prep_fitted.get_feature_names_out()
    importances = clf_fitted.feature_importances_

    feature_importance = (
        pd.DataFrame({
            'Feature': names,
            'Importancia': importances
        })
        .sort_values('Importancia', ascending=False)
        .head(25)
        .reset_index(drop=True)
    )

elif risk_champion_name == 'LogisticRegression_Balanced':
    prep_fitted = risk_champion.named_steps['prep']
    clf_fitted = risk_champion.named_steps['clf']

    names = prep_fitted.get_feature_names_out()
    coefs = np.abs(clf_fitted.coef_[0])

    feature_importance = (
        pd.DataFrame({
            'Feature': names,
            'Importancia': coefs
        })
        .sort_values('Importancia', ascending=False)
        .head(25)
        .reset_index(drop=True)
    )

if not feature_importance.empty:
    display(feature_importance.head(15).round(4))

    plot_df = feature_importance.head(12).sort_values('Importancia')

    plt.figure(figsize=(9, 5))
    plt.barh(plot_df['Feature'], plot_df['Importancia'])
    plt.title('CloudSeekers — Principais fatores associados ao risco de OLA')
    plt.xlabel('Importância relativa no modelo')
    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR / 'importancia_features_ola.png',
        dpi=180,
        bbox_inches='tight'
    )
    plt.show()

    feature_importance.to_csv(
        GOLD_RISK_DIR / 'feature_importance_ola.csv',
        index=False
    )
else:
    print('Não foi possível extrair importância das features para o modelo selecionado.')


## 25. Camada Gold — Score de Risco de OLA

Como o dataset não contém incidentes futuros ainda não abertos, o MVP utiliza o **teste fora do tempo** para demonstrar como o motor de risco funcionaria operacionalmente.

A tabela Gold contém:

- incidente;
- data de abertura;
- prioridade;
- produto/categoria/grupo;
- score estimado de risco;
- faixa de risco;
- alerta gerado pelo modelo;
- resultado real, mantido apenas para validação histórica.

Essa saída pode alimentar o **mockup do Power BI** sem necessidade de conexão real com o serviço.


In [ ]:
risk_gold = risk_test[[
    'Número',
    'Aberto',
    'Prioridade',
    'Produto',
    'Categoria',
    'Subcategoria',
    'Grupo designado',
    'Item de configuração',
    'Aberto por'
]].copy()

risk_gold['Score_Risco_OLA'] = np.round(test_prob, 6)
risk_gold['Alerta_Modelo'] = np.where(
    test_prob >= RISK_THRESHOLD,
    'SIM',
    'NAO'
)
risk_gold['KPI_Violado_Real'] = np.where(
    yt.to_numpy() == 1,
    'SIM',
    'NAO'
)
risk_gold['Periodo_Avaliacao'] = 'OUT_OF_TIME_TEST'

# Faixas relativas de priorização operacional.
q85, q95, q99 = np.quantile(test_prob, [0.85, 0.95, 0.99])

def risk_level(score):
    if score >= q99:
        return 'Crítico'
    if score >= q95:
        return 'Alto'
    if score >= q85:
        return 'Médio'
    return 'Baixo'

risk_gold['Nivel_Risco'] = [risk_level(x) for x in test_prob]

risk_gold = risk_gold.sort_values(
    ['Score_Risco_OLA', 'Aberto'],
    ascending=[False, False]
).reset_index(drop=True)

# ------------------------------------------------------------------
# GOLD PRINCIPAL — formato exato para Azure SQL
# ------------------------------------------------------------------
risco_sql = risk_gold.rename(columns={
    'Número': 'numero_incidente',
    'Aberto': 'aberto',
    'Prioridade': 'prioridade',
    'Produto': 'produto',
    'Categoria': 'categoria',
    'Grupo designado': 'grupo_designado',
    'Score_Risco_OLA': 'score_risco_ola',
    'Nivel_Risco': 'nivel_risco',
    'Alerta_Modelo': 'alerta_modelo',
    'KPI_Violado_Real': 'kpi_violado_real',
})[
    [
        'numero_incidente',
        'aberto',
        'prioridade',
        'produto',
        'categoria',
        'grupo_designado',
        'score_risco_ola',
        'nivel_risco',
        'alerta_modelo',
        'kpi_violado_real',
    ]
].copy()

risco_sql['aberto'] = pd.to_datetime(risco_sql['aberto'])
risco_sql['score_risco_ola'] = risco_sql['score_risco_ola'].astype(float)

risco_sql.to_csv(
    GOLD_RISK_DIR / 'fct_risco_ola.csv',
    index=False
)

# Mantém uma versão analítica mais rica para auditoria.
risk_gold.to_csv(
    GOLD_RISK_DIR / 'fct_risco_ola_backtest_detalhado.csv',
    index=False
)

risk_dashboard_summary = pd.DataFrame([{
    'Modelo': risk_champion_name,
    'Periodo': f'{risk_test["Aberto"].min().date()} a {risk_test["Aberto"].max().date()}',
    'Incidentes_Avaliados': len(risk_gold),
    'Alertas_Gerados': int((risk_gold['Alerta_Modelo'] == 'SIM').sum()),
    'Risco_Critico': int((risk_gold['Nivel_Risco'] == 'Crítico').sum()),
    'Risco_Alto': int((risk_gold['Nivel_Risco'] == 'Alto').sum()),
    'Violacoes_Reais': int((risk_gold['KPI_Violado_Real'] == 'SIM').sum()),
    'PR_AUC': pr_auc_test,
    'Recall_Top5pct': recall_top5,
    'Lift_Top5pct': lift_top5,
}])

risk_dashboard_summary.to_csv(
    METRICS_DIR / 'kpi_dashboard_risco_ola.csv',
    index=False
)

print('Gold Risco de OLA pronta para Azure SQL:')
display(risco_sql.head(20))

print('\nArquivo principal:')
print(GOLD_RISK_DIR / 'fct_risco_ola.csv')


## 26. DDL de Referência — Azure SQL


In [ ]:
ddl = """
-- =========================================================
-- CloudSeekers - Camada GOLD - Azure SQL
-- Estrutura alinhada aos CSVs exportados pela v4
-- =========================================================

IF NOT EXISTS (
    SELECT 1
    FROM sys.schemas
    WHERE name = 'gold'
)
BEGIN
    EXEC('CREATE SCHEMA gold');
END;
GO

CREATE TABLE gold.fct_previsao_futura (
    id_previsao        INT IDENTITY(1,1) PRIMARY KEY,
    data_referencia    DATE          NOT NULL,
    data_previsao      DATE          NOT NULL,
    prioridade         VARCHAR(20)   NOT NULL,
    horizonte_dias     INT           NOT NULL,
    volume_previsto    DECIMAL(10,2) NOT NULL,
    modelo             VARCHAR(100)  NULL,
    data_carga         DATETIME2     DEFAULT SYSDATETIME()
);
GO

CREATE INDEX ix_previsao_data
ON gold.fct_previsao_futura (data_previsao, prioridade);
GO

CREATE TABLE gold.fct_risco_ola (
    id_risco            INT IDENTITY(1,1) PRIMARY KEY,
    numero_incidente    VARCHAR(30)   NOT NULL,
    aberto              DATETIME2     NULL,
    prioridade          VARCHAR(20)   NULL,
    produto             VARCHAR(255)  NULL,
    categoria           VARCHAR(255)  NULL,
    grupo_designado     VARCHAR(255)  NULL,
    score_risco_ola     DECIMAL(8,6)  NOT NULL,
    nivel_risco         VARCHAR(20)   NOT NULL,
    alerta_modelo       VARCHAR(3)    NULL,
    kpi_violado_real    VARCHAR(3)    NULL,
    data_carga          DATETIME2     DEFAULT SYSDATETIME()
);
GO

CREATE INDEX ix_risco_score
ON gold.fct_risco_ola (score_risco_ola DESC, prioridade);
GO
""".strip()

(DOCS_DIR / 'ddl_azure_sql.sql').write_text(ddl, encoding='utf-8')
print(ddl)


## 27. Manifesto de Artefatos da v4

Ao executar o notebook do início ao fim, os principais artefatos esperados são:

### Silver
- `data/silver/incidentes_silver.parquet`

### Gold — Forecast
- `data/gold/forecast/fct_previsao_futura.csv`
- `data/gold/forecast/fct_previsao_backtest.csv`
- `data/gold/forecast/fct_previsao_proxima_semana.csv`

### Gold — Risco de OLA
- `data/gold/risco_ola/fct_risco_ola.csv`
- `data/gold/risco_ola/fct_risco_ola_backtest_detalhado.csv`
- `data/gold/risco_ola/feature_importance_ola.csv`

### Métricas
- `outputs/metrics/metricas_forecast.csv`
- `outputs/metrics/metricas_forecast_d1_d7.csv`
- `outputs/metrics/metricas_forecast_semana.csv`
- `outputs/metrics/metricas_risco_ola.csv`
- `outputs/metrics/matriz_confusao_ola.csv`

### Figuras para o PPT
- `outputs/figures/forecast_real_vs_previsto.png`
- `outputs/figures/forecast_d1_d7.png`
- `outputs/figures/matriz_confusao_ola.png`
- `outputs/figures/importancia_features_ola.png`

Os dois arquivos principais usados pelo futuro `load_gold_azure.py` são:

1. `data/gold/forecast/fct_previsao_futura.csv`
2. `data/gold/risco_ola/fct_risco_ola.csv`


In [ ]:
# Verificação final dos arquivos críticos.
critical_artifacts = {
    'Silver': SILVER_DIR / 'incidentes_silver.parquet',
    'Gold Forecast': GOLD_FORECAST_DIR / 'fct_previsao_futura.csv',
    'Gold Risco OLA': GOLD_RISK_DIR / 'fct_risco_ola.csv',
    'Métricas Forecast': METRICS_DIR / 'metricas_forecast.csv',
    'Métricas Risco OLA': METRICS_DIR / 'metricas_risco_ola.csv',
    'Figura Real x Previsto': FIGURES_DIR / 'forecast_real_vs_previsto.png',
    'Figura Forecast D+1-D+7': FIGURES_DIR / 'forecast_d1_d7.png',
    'Figura Matriz de Confusão': FIGURES_DIR / 'matriz_confusao_ola.png',
}

artifact_check = pd.DataFrame([
    {
        'Artefato': name,
        'Caminho': str(path),
        'Existe': path.exists(),
        'Tamanho_KB': round(path.stat().st_size / 1024, 1) if path.exists() else None,
    }
    for name, path in critical_artifacts.items()
])

display(artifact_check)

if artifact_check['Existe'].all():
    print('✓ Todos os artefatos críticos da v4 foram gerados.')
else:
    print('Atenção: há artefatos ainda não gerados. Execute todas as células anteriores.')


## 28. Resumo Técnico para a Sprint 3

A versão v4 consolida o pipeline técnico do CloudSeekers:

1. ingestão e preservação do dataset original na **Bronze**;
2. ETL reproduzível em **Python/Pandas**;
3. validações de Data Quality;
4. publicação da **Silver** em Parquet;
5. diagnóstico temporal e mudança de regime;
6. feature engineering sem leakage;
7. comparação de baselines, Random Forest e XGBoost;
8. validação temporal por rolling-origin backtest;
9. Forecast P2/P3 para **D+1 até D+7**;
10. exportação da Gold `fct_previsao_futura.csv` pronta para Azure SQL;
11. classificação do **risco de violação de OLA**;
12. tratamento do desbalanceamento da classe;
13. validação por PR-AUC, Recall, Precision, F1, F2 e Lift@5%;
14. explicabilidade do modelo;
15. exportação da Gold `fct_risco_ola.csv` pronta para Azure SQL;
16. geração automática de métricas e figuras para evidência no PPT;
17. DDL alinhado ao schema `gold` do Azure SQL.

### Separação de responsabilidades técnicas

- **Notebook v4:** Data Engineering + Data Science + geração das Golds.
- **`load_gold_azure.py`:** leitura dos CSVs Gold e persistência no Azure SQL.
- **Dashboard:** mockup visual alimentado por números coerentes com os resultados do MVP.

Essa separação permite demonstrar um pipeline real e organizado sem armazenar credenciais do Azure dentro do notebook.
